# Single Task Visualization by Corner - Adam vs Grad&Move

This notebook compares Adam adaptation vs Grad&Move adaptation for each corner.

**Task Definition per Corner:**
- FF: 4 support → 2 test (grad/move meaningful)
- SS: 4 support → 2 test (grad/move meaningful)
- TT: 2 support → 1 test (grad/move perfect fit)

**Grad & Move Formula:**
- grad = (y_max - y_min) / (pred_max - pred_min)
- move = center - y_middle / grad
- adjusted = (pred - move) * grad

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import random
import re
import matplotlib.pyplot as plt
from pathlib import Path
from collections import OrderedDict, defaultdict

# Add paths
sys.path.insert(0, '/home/tkdgn2907/Deepsets_test/MAML/Projects/model_code')
from maml_optimized import OptimizedMAML, MAMLModel_3hidden

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DATA_DIR = '/home/tkdgn2907/Deepsets_test/MAML/Projects/CAD_TEST/AND_cells_extracted'
DATA_TYPE = 'cell'
MODEL_PATH = '/home/tkdgn2907/Deepsets_test/MAML/Projects/pretrained_models/training_loss_taskdivide_all/cell_innerdiv100_meta32_combined_519traintask_full1DMAML_weights_3hidden_(40)_300000_inner1_upgraded_tsmc.pth'
TSMC_TRAIN_INPUT_PATH = '/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/MLP_dataset_TSMC/combined_data/tsmc_topology_agnostic_train_input_cell.pth'

GPU_ID = '0'
RANDOM_TASK_ID = None  # Set to specific number or None for random
MAX_STEPS = 100
LOSS_THRESHOLD = 0.001  # Threshold for Adam activation in grad_move

# Corner splits
CORNER_SPLITS = {
    'FF': {
        'support': ['ff0p88vm40c', 'ff0p99v125c', 'ff1p1v125c', 'ff1p1vm40c'],
        'test': ['ff0p88v125c', 'ff0p99vm40c']
    },
    'SS': {
        'support': ['ss0p72vm40c', 'ss0p81v125c', 'ss0p9v125c', 'ss0p9vm40c'],
        'test': ['ss0p72v125c', 'ss0p81vm40c']
    },
    'TT': {
        'support': ['tt0p8v25c', 'tt0p9v25c'],
        'test': ['tt1p0v25c']
    }
}

# TSMC Process Parameters
PARAM_A = [1.427, 1.457, 1.430, 1.470, 1.443, 1.483, 1.43, 1.47, 1.43, 1.47]
PARAM_B = [0.026, 0.045, 0, 0, -0.026, -0.05, 0.0208, -0.04, 0.036, -0.0208]
PARAM_C = [0.024, 2.000, 0.024, 2.000, 0.024, 2.000, 0.024, 2.000, 0.024, 2.000]
CORNER_TO_IDX = {'FF': 0, 'TT': 1, 'SS': 2, 'FS': 3, 'SF': 4}

In [ ]:
# GPU settings
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {device}')

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def parse_filename(filename):
    match = re.search(r'(ff|tt|ss|fs|sf)(\d+p\d+)v(m?\d+)c', filename.lower())
    if not match:
        raise ValueError(f"Cannot parse filename: {filename}")
    corner = match.group(1).upper()
    voltage = float(match.group(2).replace('p', '.'))
    temp_str = match.group(3)
    temperature = -float(temp_str[1:]) if temp_str.startswith('m') else float(temp_str)
    return corner, voltage, temperature

def get_abc_params(corner):
    idx = CORNER_TO_IDX.get(corner.upper(), 1)
    nmos_idx, pmos_idx = idx * 2, idx * 2 + 1
    return {'a_n': PARAM_A[nmos_idx], 'a_p': PARAM_A[pmos_idx],
            'b_n': PARAM_B[nmos_idx], 'b_p': PARAM_B[pmos_idx],
            'c_n': PARAM_C[nmos_idx], 'c_p': PARAM_C[pmos_idx]}

def parse_lib_file(lib_path):
    with open(lib_path, 'r') as f:
        content = f.read()
    samples = []
    for cell_match in re.finditer(r'cell\s*\((\w+)\)\s*\{', content):
        cell_name = cell_match.group(1)
        cell_start = cell_match.end()
        brace_count = 1
        for i in range(cell_start, len(content)):
            if content[i] == '{': brace_count += 1
            elif content[i] == '}':
                brace_count -= 1
                if brace_count == 0:
                    cell_end = i
                    break
        cell_content = content[cell_start:cell_end]
        for timing_match in re.finditer(r'timing\s*\(\)\s*\{\s*related_pin\s*:\s*"(\w+)"', cell_content):
            related_pin = timing_match.group(1)
            timing_start = timing_match.end()
            t_brace = 1
            for i in range(timing_start, len(cell_content)):
                if cell_content[i] == '{': t_brace += 1
                elif cell_content[i] == '}':
                    t_brace -= 1
                    if t_brace == 0:
                        timing_end = i
                        break
            timing_content = cell_content[timing_start:timing_end]
            for delay_type in ['cell_rise', 'cell_fall']:
                table_match = re.search(rf'{delay_type}\s*\([^)]+\)\s*\{{', timing_content)
                if table_match:
                    table_start = table_match.end()
                    tb = 1
                    for i in range(table_start, len(timing_content)):
                        if timing_content[i] == '{': tb += 1
                        elif timing_content[i] == '}':
                            tb -= 1
                            if tb == 0:
                                table_end = i
                                break
                    table_content = timing_content[table_start:table_end]
                    idx1 = re.search(r'index_1\s*\(\s*"([^"]+)"\s*\)', table_content)
                    idx2 = re.search(r'index_2\s*\(\s*"([^"]+)"\s*\)', table_content)
                    vals = re.search(r'values\s*\(\s*(.*?)\s*\)\s*;', table_content, re.DOTALL)
                    if idx1 and idx2 and vals:
                        index_1 = [float(x.strip()) for x in idx1.group(1).split(',')]
                        index_2 = [float(x.strip()) for x in idx2.group(1).split(',')]
                        values_str = vals.group(1).replace('\\', '').replace('\n', ' ')
                        rows = re.findall(r'"([^"]+)"', values_str)
                        values = [[float(x.strip()) for x in row.split(',')] for row in rows]
                        samples.append({'cell_name': cell_name, 'delay_type': delay_type,
                                       'related_pin': related_pin, 'index_1': index_1,
                                       'index_2': index_2, 'values': values})
    return samples

def create_mlp_input(samples, corner, voltage, temperature):
    abc_params = get_abc_params(corner)
    inputs, outputs, metadata = [], [], []
    for sample in samples:
        if sample['delay_type'] not in ['cell_rise', 'cell_fall']:
            continue
        delay_indicator = -1 if 'rise' in sample['delay_type'] else 1
        a = (abc_params['a_n'] + abc_params['a_p']) / 2
        b = abc_params['b_n'] + abc_params['b_p']
        c = abc_params['c_n'] + abc_params['c_p']
        for ri, slew in enumerate(sample['index_1']):
            for ci, load in enumerate(sample['index_2']):
                if ri < len(sample['values']) and ci < len(sample['values'][ri]):
                    inputs.append([a, b, c, temperature, voltage, 2, delay_indicator, slew, load])
                    outputs.append([sample['values'][ri][ci]])
                    metadata.append({'cell_name': sample['cell_name'], 'delay_type': sample['delay_type'],
                                    'related_pin': sample['related_pin'], 'slew_idx': ri, 'load_idx': ci,
                                    'slew': slew, 'load': load})
    return (torch.tensor(inputs, dtype=torch.float32),
            torch.tensor(outputs, dtype=torch.float32), metadata) if inputs else (torch.tensor([]), torch.tensor([]), [])

In [ ]:
# ============================================================
# LOAD ALL DATA
# ============================================================

print("Loading data...")
base_path = Path(DATA_DIR)
all_data = {}

for voltage_dir in ['0p8v', '0p9v', '1p0v']:
    dir_path = base_path / voltage_dir
    if not dir_path.exists():
        continue
    for lib_file in sorted(dir_path.glob('AND_*.tlib')):
        match = re.search(r'AND_lib1_(\w+)_base_400\.tlib', lib_file.name)
        if not match:
            continue
        condition = match.group(1)
        corner, voltage, temperature = parse_filename(lib_file.name)
        samples = parse_lib_file(str(lib_file))
        inputs, outputs, metadata = create_mlp_input(samples, corner, voltage, temperature)
        if len(inputs) > 0:
            all_data[condition] = (inputs, outputs, metadata, corner, voltage, temperature)
            print(f"  {condition}: {len(inputs)} samples ({corner}, {voltage}V, {temperature}C)")

print(f"\nTotal conditions loaded: {len(all_data)}")

In [ ]:
# ============================================================
# APPLY TSMC NORMALIZATION
# ============================================================

print("Loading TSMC normalization stats...")
tsmc_train = torch.load(TSMC_TRAIN_INPUT_PATH)
norm_indices = [3, 4, 7, 8]
feature_names = {3: 'temperature', 4: 'voltage', 7: 'slew', 8: 'load'}
tsmc_norm_stats = {}

print("\nTSMC Normalization Stats:")
for idx in norm_indices:
    mean = tsmc_train[:, :, idx].mean().item()
    std = tsmc_train[:, :, idx].std().item()
    tsmc_norm_stats[idx] = (mean, std)
    print(f"  {feature_names[idx]}: mean={mean:.6f}, std={std:.6f}")
del tsmc_train

def apply_norm(data, stats):
    for idx, (mean, std) in stats.items():
        if std > 0:
            data[:, idx] = (data[:, idx] - mean) / std

print("\nApplying normalization...")
for cond in all_data:
    inputs, outputs, metadata, corner, voltage, temp = all_data[cond]
    apply_norm(inputs, tsmc_norm_stats)
print("Done!")

In [ ]:
# ============================================================
# BUILD TASK INDEX
# ============================================================

def build_index(data_dict):
    index = defaultdict(lambda: defaultdict(dict))
    for cond, (inputs, outputs, metadata, corner, voltage, temp) in data_dict.items():
        for idx, meta in enumerate(metadata):
            cell_key = (meta['cell_name'], meta['delay_type'], meta['related_pin'])
            sl_key = (meta['slew_idx'], meta['load_idx'])
            index[cell_key][cond][sl_key] = idx
    return index

data_index = build_index(all_data)
print(f"Total unique (cell, delay, pin) combinations: {len(data_index)}")

In [ ]:
# ============================================================
# LOAD MODEL
# ============================================================

print(f"Loading model: {MODEL_PATH}")
maml_model = OptimizedMAML(
    model=MAMLModel_3hidden(in_features=9, layer_length=40),
    dataset_in=None, dataset_out=None, inner_lr=0.001, meta_lr=0.0001
)
state_dict = torch.load(MODEL_PATH, map_location=device)
maml_model.model.load_state_dict(state_dict)
maml_model.model.to(device)
maml_model.model.eval()
print("Model loaded!")

In [ ]:
# ============================================================
# SELECT RANDOM TASK
# ============================================================

# Find a task that exists in all conditions
all_conditions = set()
for split in CORNER_SPLITS.values():
    all_conditions.update(split['support'])
    all_conditions.update(split['test'])

valid_tasks = []
for cell_key in data_index:
    sl_keys = None
    for cond in all_conditions:
        if cond in data_index[cell_key]:
            cond_sl = set(data_index[cell_key][cond].keys())
            sl_keys = cond_sl if sl_keys is None else sl_keys & cond_sl
    if sl_keys:
        for sl_key in sl_keys:
            if all(cond in data_index[cell_key] and sl_key in data_index[cell_key][cond] 
                   for cond in all_conditions):
                valid_tasks.append((cell_key, sl_key))

print(f"Total valid tasks across all corners: {len(valid_tasks)}")

if RANDOM_TASK_ID is None:
    task_idx = random.randint(0, len(valid_tasks) - 1)
else:
    task_idx = RANDOM_TASK_ID

cell_key, sl_key = valid_tasks[task_idx]

print("\n" + "=" * 80)
print(f"Selected Task: {task_idx} / {len(valid_tasks)}")
print("=" * 80)
print(f"  Cell: {cell_key[0]}")
print(f"  Delay: {cell_key[1]}")
print(f"  Pin: {cell_key[2]}")
print(f"  Slew idx: {sl_key[0]}, Load idx: {sl_key[1]}")

In [ ]:
# ============================================================
# ADAPTATION FUNCTIONS
# ============================================================

def run_adaptation_adam(initial_model, X_support, y_support, X_query, y_query,
                        max_steps=100, lr=3e-3):
    """Adam weight adaptation with tracking"""
    model = nn.Sequential(OrderedDict([
        ('l1', nn.Linear(9, 40)), ('relu1', nn.ReLU()),
        ('l2', nn.Linear(40, 40)), ('relu3', nn.ReLU()),
        ('l4', nn.Linear(40, 40)), ('relu2', nn.ReLU()),
        ('l3', nn.Linear(40, 1))
    ])).to(device)
    model.load_state_dict(initial_model.state_dict())

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    y_mean, y_std = y_support.mean(), y_support.std()
    if y_std < 1e-8:
        y_std = torch.tensor(1.0)
    y_target = (y_support - y_mean) / y_std

    # NRMSE calculation
    y_query_flat = y_query.flatten()
    y_range = y_query_flat.max() - y_query_flat.min()
    if y_range < 1e-8:
        y_range = y_query_flat.mean()  # Use mean for single sample

    support_losses, query_nrmses = [], []
    predictions_over_time = []

    for step in range(max_steps + 1):
        with torch.no_grad():
            pred_support = model(X_support)
            support_loss = criterion(pred_support, y_target).item()
            support_losses.append(support_loss)

            pred_query = model(X_query) * y_std + y_mean
            pred_query_flat = pred_query.flatten()
            
            # NRMSE
            rmse = torch.sqrt(torch.mean((pred_query_flat - y_query_flat) ** 2))
            nrmse = (rmse / (y_range + 1e-8)).item() * 100
            query_nrmses.append(nrmse)
            predictions_over_time.append(pred_query.cpu().numpy().flatten())

        if step < max_steps:
            model.zero_grad()
            loss = criterion(model(X_support), y_target)
            loss.backward()
            optimizer.step()

    return {
        'support_losses': support_losses,
        'query_nrmses': query_nrmses,
        'predictions_over_time': predictions_over_time,
        'actual_values': y_query.cpu().numpy().flatten()
    }


def run_adaptation_grad_move(initial_model, X_support, y_support, X_query, y_query,
                              max_steps=100, lr=3e-3, loss_threshold=0.001):
    """Grad and Move: 단순 선형 보정 후, Adam adaptation (always runs)
    
    보정 공식: adjusted = (pred - move) * grad
    - grad = (y_max - y_min) / (pred_max - pred_min)
    - center = (pred_max + pred_min) / 2
    - y_middle = (y_max + y_min) / 2
    - move = center - y_middle / grad
    """
    model = nn.Sequential(OrderedDict([
        ('l1', nn.Linear(9, 40)), ('relu1', nn.ReLU()),
        ('l2', nn.Linear(40, 40)), ('relu3', nn.ReLU()),
        ('l4', nn.Linear(40, 40)), ('relu2', nn.ReLU()),
        ('l3', nn.Linear(40, 1))
    ])).to(device)
    model.load_state_dict(initial_model.state_dict())

    y_support_flat = y_support.flatten()
    y_query_flat = y_query.flatten()
    criterion = nn.MSELoss()

    # NRMSE calculation
    y_range = y_query_flat.max() - y_query_flat.min()
    if y_range < 1e-8:
        y_range = y_query_flat.mean()  # Use mean for single sample

    # ============================================================
    # Step 1: Grad & Move 계산 (단순 선형 보정)
    # ============================================================
    model.eval()
    with torch.no_grad():
        pred_support = model(X_support).flatten()
        pred_query = model(X_query).flatten()

    # 예측값/실제값 범위
    pred_min = pred_support.min().item()
    pred_max = pred_support.max().item()
    y_min = y_support_flat.min().item()
    y_max = y_support_flat.max().item()

    # Use average of min/max instead of middle index
    center = (pred_max + pred_min) / 2
    y_middle = (y_max + y_min) / 2

    # grad (scale)
    if abs(pred_max - pred_min) > 1e-8:
        grad = (y_max - y_min) / (pred_max - pred_min)
    else:
        grad = 1.0

    # move (offset)
    if abs(grad) > 1e-8:
        move = center - y_middle / grad
    else:
        move = 0.0

    # 보정된 예측값: (pred - move) * grad
    adjusted_support = (pred_support - move) * grad
    adjusted_query = (pred_query - move) * grad

    initial_loss = criterion(adjusted_support, y_support_flat).item()
    
    # Initial NRMSE
    rmse = torch.sqrt(torch.mean((adjusted_query - y_query_flat) ** 2))
    initial_nrmse = (rmse / (y_range + 1e-8)).item() * 100

    support_losses = [initial_loss]
    query_nrmses = [initial_nrmse]
    predictions_over_time = [adjusted_query.cpu().numpy()]

    # ============================================================
    # Step 2: Always run Adam adaptation (threshold >= 0)
    # ============================================================
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    y_mean, y_std = y_support_flat.mean(), y_support_flat.std()
    if y_std < 1e-8:
        y_std = torch.tensor(1.0)
    y_target = (y_support_flat - y_mean) / y_std

    for step in range(max_steps):
        # 기록 먼저
        with torch.no_grad():
            pred_s = model(X_support).flatten()
            pred_q = model(X_query).flatten() * y_std + y_mean

            support_loss = criterion(pred_s, y_target).item()
            support_losses.append(support_loss)

            # NRMSE
            rmse = torch.sqrt(torch.mean((pred_q - y_query_flat) ** 2))
            nrmse = (rmse / (y_range + 1e-8)).item() * 100
            query_nrmses.append(nrmse)
            predictions_over_time.append(pred_q.cpu().numpy())

        # 학습
        model.zero_grad()
        loss = criterion(model(X_support).flatten(), y_target)
        loss.backward()
        optimizer.step()

    return {
        'support_losses': support_losses,
        'query_nrmses': query_nrmses,
        'predictions_over_time': predictions_over_time,
        'actual_values': y_query_flat.cpu().numpy(),
        'grad': grad,
        'move': move,
        'adam_used': True,  # Always True now
        'initial_loss': initial_loss
    }

In [ ]:
# ============================================================
# RUN ADAPTATION FOR EACH CORNER
# ============================================================

corner_results = {}

for corner_name, split in CORNER_SPLITS.items():
    print(f"\n{'='*60}")
    print(f"Processing {corner_name} Corner")
    print(f"  Support: {split['support']} ({len(split['support'])} samples)")
    print(f"  Test: {split['test']} ({len(split['test'])} samples)")
    print(f"{'='*60}")
    
    # Build support set
    X_support_list, y_support_list, support_info = [], [], []
    for cond in split['support']:
        inputs, outputs, metadata, corner, voltage, temp = all_data[cond]
        sample_idx = data_index[cell_key][cond][sl_key]
        X_support_list.append(inputs[sample_idx:sample_idx+1])
        y_support_list.append(outputs[sample_idx:sample_idx+1])
        support_info.append({'condition': cond, 'voltage': voltage, 'temp': temp, 
                            'output': outputs[sample_idx].item()})
    
    # Build query set
    X_query_list, y_query_list, query_info = [], [], []
    for cond in split['test']:
        inputs, outputs, metadata, corner, voltage, temp = all_data[cond]
        sample_idx = data_index[cell_key][cond][sl_key]
        X_query_list.append(inputs[sample_idx:sample_idx+1])
        y_query_list.append(outputs[sample_idx:sample_idx+1])
        query_info.append({'condition': cond, 'voltage': voltage, 'temp': temp,
                          'output': outputs[sample_idx].item()})
    
    X_support = torch.cat(X_support_list, dim=0).to(device)
    y_support = torch.cat(y_support_list, dim=0).to(device)
    X_query = torch.cat(X_query_list, dim=0).to(device)
    y_query = torch.cat(y_query_list, dim=0).to(device)
    
    print(f"\n  Support:")
    for info in support_info:
        print(f"    {info['condition']}: {info['voltage']}V, {info['temp']}C -> {info['output']:.6f} ns")
    
    print(f"\n  Query:")
    for info in query_info:
        print(f"    {info['condition']}: {info['voltage']}V, {info['temp']}C -> {info['output']:.6f} ns")
    
    # Run both adaptation methods
    adam_results = run_adaptation_adam(
        maml_model.model.model, X_support, y_support, X_query, y_query, max_steps=MAX_STEPS
    )
    gm_results = run_adaptation_grad_move(
        maml_model.model.model, X_support, y_support, X_query, y_query, 
        max_steps=MAX_STEPS, loss_threshold=LOSS_THRESHOLD
    )
    
    corner_results[corner_name] = {
        'adam': adam_results,
        'grad_move': gm_results,
        'support_info': support_info,
        'query_info': query_info
    }
    
    # Print results
    print(f"\n  Results (step 40):")
    print(f"    Adam NRMSE:      {adam_results['query_nrmses'][40]:.2f}%")
    print(f"    Grad&Move NRMSE: {gm_results['query_nrmses'][40]:.2f}%")
    print(f"    Grad&Move info: grad={gm_results['grad']:.4f}, move={gm_results['move']:.4f}")
    print(f"    Grad&Move initial_loss: {gm_results['initial_loss']:.2e}")
    print(f"    Adam used: {gm_results['adam_used']}")

In [ ]:
# ============================================================
# VISUALIZATION: NRMSE CURVES (Adam vs Grad&Move)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = {'FF': 'green', 'SS': 'red', 'TT': 'blue'}

for idx, (corner_name, data) in enumerate(corner_results.items()):
    ax = axes[idx]
    
    ax.plot(data['adam']['query_nrmses'], color=colors[corner_name], 
            linewidth=1.5, label='Adam')
    ax.plot(data['grad_move']['query_nrmses'], color=colors[corner_name], 
            linewidth=1.5, linestyle='--', label='Grad&Move')
    
    ax.axvline(x=40, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Step')
    ax.set_ylabel('NRMSE (%)')
    
    n_support = len(data['support_info'])
    adam_used = data['grad_move']['adam_used']
    ax.set_title(f'{corner_name} (Support: {n_support})\nAdam used: {adam_used}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)

plt.suptitle(f'NRMSE Curves - Adam vs Grad&Move\nTask: {cell_key[0]} / {cell_key[1]}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VISUALIZATION: PREDICTIONS VS ACTUAL
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (corner_name, data) in enumerate(corner_results.items()):
    ax = axes[idx]
    query_info = data['query_info']
    
    x_labels = [info['condition'] for info in query_info]
    x_pos = np.arange(len(x_labels))
    
    actuals = data['adam']['actual_values']
    adam_preds = data['adam']['predictions_over_time'][40]
    gm_preds = data['grad_move']['predictions_over_time'][40]
    
    width = 0.25
    ax.bar(x_pos - width, actuals, width, label='Actual', color='black', alpha=0.7)
    ax.bar(x_pos, adam_preds, width, label='Adam@40', color=colors[corner_name], alpha=0.7)
    ax.bar(x_pos + width, gm_preds, width, label='G&M@40', color=colors[corner_name], alpha=0.4, hatch='//')
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Delay (ns)')
    ax.set_title(f'{corner_name} Corner')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Predictions vs Actual by Corner', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SUMMARY TABLE
# ============================================================

print("\n" + "=" * 100)
print("SUMMARY: Adam vs Grad&Move")
print("=" * 100)
print(f"\nTask: {cell_key[0]} / {cell_key[1]} / {cell_key[2]}")
print(f"Slew idx: {sl_key[0]}, Load idx: {sl_key[1]}")
print(f"Loss Threshold: {LOSS_THRESHOLD}")
print()
print(f"{'Corner':<10} {'Support':<10} {'Test':<10} {'Adam@40':<12} {'G&M@40':<12} {'G&M Adam?':<12} {'G&M Loss':<15}")
print("-" * 85)

for corner_name, data in corner_results.items():
    n_support = len(data['support_info'])
    n_test = len(data['query_info'])
    adam_nrmse = data['adam']['query_nrmses'][40]
    gm_nrmse = data['grad_move']['query_nrmses'][40]
    adam_used = data['grad_move']['adam_used']
    init_loss = data['grad_move']['initial_loss']
    print(f"{corner_name:<10} {n_support:<10} {n_test:<10} {adam_nrmse:<12.2f} {gm_nrmse:<12.2f} {str(adam_used):<12} {init_loss:<15.2e}")

In [ ]:
# ============================================================
# ANALYSIS: Grad&Move vs Adam Comparison
# ============================================================

print("\n" + "=" * 80)
print("ANALYSIS: Grad&Move Formula")
print("=" * 80)
print()
print("Grad&Move uses linear transformation before Adam:")
print("  - center = (pred_max + pred_min) / 2")
print("  - y_middle = (y_max + y_min) / 2")
print("  - grad = (y_max - y_min) / (pred_max - pred_min)")
print("  - move = center - y_middle / grad")
print("  - adjusted = (pred - move) * grad")
print()
print("Both methods now run Adam adaptation after initial step.")
print("The difference is that Grad&Move applies linear correction first.")